In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
from datetime import datetime as dt
from metpy.units import units
import metpy.calc as mpcalc
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt

In [ ]:
%%time 

ds = xr.open_dataset(
    "https://data.earthdatahub.destine.eu/era5/reanalysis-era5-pressure-levels-v0.zarr",
    storage_options={"client_kwargs":{"trust_env":True}},
    chunks={},
    engine="zarr",
)

print(f'size: {ds.nbytes / (1024 ** 4)} TB')

In [ ]:
lonW = -77.7
lonE = -73.7
latS = 42.6
latN = 44.6
cLat, cLon = (latS + latN)/2, (lonW + lonE)/2 

# Recall that in ERA5, longitudes run between 0 and 360, not -180 and 180
if (lonW < 0 ):
    lonW = lonW + 360
if (lonE < 0 ):
    lonE = lonE + 360
    
expand = 1
latRange = np.arange(latS - expand,latN + expand,.25) # expand the data range a bit beyond the plot range
lonRange = np.arange((lonW - expand),(lonE + expand),.25) # Need to match longitude values to those of the coordinate variable

# Date/Time specification; I was born on October 5th, 2000 at, again according to my mom, around 12:30 PM. 
Year = 2023
Month = 8
Day = 7
Hour = 23
Minute = 0
dateTime = dt(Year,Month,Day,Hour)
timeStr = dateTime.strftime("%Y-%m-%d %H%M UTC")
timeStr

In [ ]:
ds

In [ ]:
%%time 
use_DE = True
if (use_DE):
    v = ds['v'].sel(valid_time=dateTime,latitude=latRange,longitude=lonRange, method='nearest')
else:
    v = ds['northward_wind'].sel(time=dateTime,latitude=latRange,longitude=lonRange)

v

In [ ]:
print(v.values)

In [ ]:
target_pressure = 1000 * units.hPa 
v_selected_level = v.sel(isobaricInhPa=target_pressure, method='nearest')

print("Shape after selecting 1000 hPa level:", v_selected_level.shape)

In [ ]:
datacrs = ccrs.PlateCarree()

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(1, 1, 1, projection=datacrs) # Or datacrs

ax.set_extent([lonW, lonE, latS, latN], crs=datacrs)
ax.add_feature(cfeature.STATES.with_scale('50m'), edgecolor='black', alpha=0.6)

gl = ax.gridlines(draw_labels=True, linewidth=1, color='gray', alpha=0.5, linestyle='--')
gl.top_labels = False
gl.right_labels = False

# This is the line that generates the error
cf = ax.contourf(v_selected_level.longitude, v_selected_level.latitude, v_selected_level,
                 levels=15, cmap='coolwarm', transform=datacrs)

# Add a colorbar
cbar = fig.colorbar(cf, ax=ax, orientation='vertical', 
                    label=f'V Wind Component ({v_selected_level.metpy.units})') 

# Add a title
level_val = v_selected_level.isobaricInhPa.item() # Get the exact value of the level
ax.set_title(f'V Wind Component at {level_val:.0f} hPa for {timeStr}')

plt.show()

In [ ]:
if (use_DE):
    vert = ds['w'].sel(valid_time=dateTime,latitude=latRange,longitude=lonRange, method='nearest')
else:
    vert = ds['lagrangian_tendency_of_air_pressure'].sel(time=dateTime,latitude=latRange,longitude=lonRange)

vert

In [ ]:
vert_hpa = vert.metpy.convert_units('hPa/s')

In [ ]:
print(vert_hpa.values)

In [ ]:
target_pressure = 500 * units.hPa 
vert_selected_level = vert_hpa.sel(isobaricInhPa=target_pressure, method='nearest')

print("Shape after selecting 500 hPa level:", vert_selected_level.shape) # Expected: (16, 24)

In [ ]:
datacrs = ccrs.PlateCarree()

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(1, 1, 1, projection=datacrs) # Or datacrs

ax.set_extent([lonW, lonE, latS, latN], crs=datacrs)
ax.add_feature(cfeature.STATES.with_scale('50m'), edgecolor='black', alpha=0.6)

gl = ax.gridlines(draw_labels=True, linewidth=1, color='gray', alpha=0.5, linestyle='--')
gl.top_labels = False
gl.right_labels = False

# This is the line that generates the error
cf = ax.contourf(vert_selected_level.longitude, vert_selected_level.latitude, vert_selected_level,
                 levels=15, cmap='coolwarm', transform=datacrs)

# Add a colorbar
cbar = fig.colorbar(cf, ax=ax, orientation='vertical', 
                    label=f'Vertical Velocity ({vert_selected_level.metpy.units})') 

# Add a title
level_val = vert_selected_level.isobaricInhPa.item() # Get the exact value of the level
ax.set_title(f'Vertical Pressure Velocity at {level_val:.0f} hPa for {timeStr}')

plt.show()